# Legal Information Assistant Notebook

In [4]:
!pip install langchain langchain_community
!pip install langchain_huggingface faiss-cpu
!pip install langchain-google-genai pypdf presidio-analyzer
!pip install presidio-anonymizer rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━

## 1. Environment Setup & Library Installations

This section installs all the necessary Python libraries required for this project, including `langchain`, `faiss-cpu`, `langchain-google-genai`, `pypdf`, `presidio-analyzer`, `presidio-anonymizer`, and `rank_bm25`.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

PDF_FOLDER = "/content/legal_pdfs"

all_documents = []

# Load all PDFs
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"):
        filepath = os.path.join(PDF_FOLDER, filename)

        print(f"Loading {filename}...")

        loader = PyPDFLoader(filepath)
        docs = loader.load()

        # Add custom metadata
        for doc in docs:
            doc.metadata["filename"] = filename
            doc.metadata["document_type"] = "legal_act"

            # Identify source law
            lower_name = filename.lower()

            if "constitution" in lower_name:
                doc.metadata["law"] = "Constitution of India"

            elif "nyaya" in lower_name:
                doc.metadata["law"] = "Bharatiya Nyaya Sanhita"

            elif "nagarik" in lower_name:
                doc.metadata["law"] = "Bharatiya Nagarik Suraksha Sanhita"

            elif "sakshya" in lower_name:
                doc.metadata["law"] = "Bharatiya Sakshya Adhiniyam"

            elif "it" in lower_name:
                doc.metadata["law"] = "Information Technology Act"

            elif "precedent" in lower_name:
                doc.metadata["law"] = "Precedent Cases"

            elif "consumer" in lower_name:
                doc.metadata["law"] = "Consumer Protection Act"

            elif "motor" in lower_name:
                doc.metadata["law"] = "Motor Vehicles Act"

            else:
                doc.metadata["law"] = "Unknown"

        all_documents.extend(docs)

print(f"\nTotal pages loaded: {len(all_documents)}")

/tmp/ipykernel_5403/3108387754.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading Bharatiya Nyaya Sanhita.pdf...
Loading Consumer Protection Act 2019.pdf...
Loading Constitution of India.pdf...
Loading Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf...
Loading PRECEDENT.pdf...
Loading Bharatiya Sakshya Adhiniyam 2023.pdf...
Loading Motor Vehicles Act 1988.pdf...
Loading it_act_2000_updated.pdf...

Total pages loaded: 1099


## 2. Data Loading and Preprocessing

Here, we load legal documents from PDF files, clean their text content, and prepare them for embedding. Each document is enriched with metadata like filename, document type, and the law it pertains to.

In [ ]:
import re

def clean_legal_text(text):
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove page numbering
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text)

    # Remove standalone page numbers
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)

    # Remove repeated newlines
    text = re.sub(r'\n+', '\n', text)

    return text.strip()

for doc in all_documents:
    doc.page_content = clean_legal_text(doc.page_content)

In [ ]:
def validate_query(query):
    query = query.strip()

    banned = [
        "ignore previous instructions",
        "system prompt",
        "reveal prompt",
        "jailbreak"
    ]

    for phrase in banned:
        if phrase.lower() in query.lower():
            raise ValueError("Invalid query")

    return query

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = splitter.split_documents(all_documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 5541


### Text Chunking

Legal documents can be very long, so we split them into smaller, manageable chunks. This helps the retrieval system focus on relevant passages and improves the quality of responses. We use `RecursiveCharacterTextSplitter` to create overlapping chunks for better context retention.

In [ ]:
for idx, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = idx

In [ ]:
print(chunks[0].metadata)

print("\nChunk Preview:\n")
print(chunks[0].page_content[:500])

{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': '/content/legal_pdfs/Bharatiya Nyaya Sanhita.pdf', 'total_pages': 102, 'page': 0, 'page_label': '1', 'filename': 'Bharatiya Nyaya Sanhita.pdf', 'document_type': 'legal_act', 'law': 'Bharatiya Nyaya Sanhita', 'chunk_id': 0}

Chunk Preview:

THE BHARA TIY A NY A Y A SANHITA, 2023 NO. 45 OF 2023 [25th December ,2023.] An Act to consolidate and amend the provisions relating to offences and for matters connected therewith or incidental thereto. BE it enacted by Parliament in the Seventy-fourth Y ear of the Republic of India as follows:–– CHAPTER I PRELIMINARY 1.(1) This Act may be called the Bharatiya Nyaya Sanhita, 2023. (2) It shall come into force on such date as the Central Government may , by notification in the Official Gazette, 


In [ ]:
import pickle

with open("legal_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved successfully")

Chunks saved successfully


### Saving and Loading Processed Chunks

To avoid reprocessing the documents every time, the chunks are saved to a pickle file (`legal_chunks.pkl`). This allows for quick loading in subsequent sessions. The next cell loads these pre-processed chunks.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"}
)

# Test
vector = embeddings.embed_query("What is RAG?")

print(f"Vector size: {len(vector)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Vector size: 1024


### API Key Setup (Securely using Colab Secrets)

For security reasons, especially when sharing notebooks on GitHub, it is highly recommended to store API keys using Colab's secret management.

**To set up your Gemini API Key:**
1. Click the '🔑' icon (Secrets) in the left sidebar of your Colab notebook.
2. Click 'Add new secret'.
3. For 'Name', enter `GOOGLE_API_KEY`.
4. For 'Value', paste your actual Gemini API key.
5. Ensure 'Notebook access' is toggled on.

The following code will then securely fetch your API key.

In [6]:
import os
from google.colab import userdata

# Fetch the API key securely from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Set it as an environment variable for consistent access
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Google API Key loaded securely.")

Google API Key loaded securely.


In [7]:
import pickle

with open("legal_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

## 3. Embedding and Vector Store Creation

This section focuses on converting text chunks into numerical vector representations (embeddings) and storing them in a vector database (FAISS) for efficient similarity search.

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    chunks,
    embeddings
)

db.save_local("vidhi_faiss")

KeyboardInterrupt: 

### Creating/Loading the FAISS Vector Store

The FAISS vector store indexes the embedded chunks, allowing for fast similarity searches. This cell either creates a new FAISS index from the embedded chunks and saves it locally, or loads an existing one if the notebook is rerun. Note that `0t66NMtxzbYu` will create and save the `vidhi_faiss` index, and `LTFRVC4vCIAF` will load it for subsequent uses.

In [10]:
from langchain_community.vectorstores import FAISS
folder_path = "/content/vidhi_faiss"

# 3. Load the folder directory as the 'db' object
db = FAISS.load_local(
    folder_path=folder_path,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

/tmp/ipykernel_1487/3331006157.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 4. Retrieval System Setup

This section configures the retrieval mechanisms, combining both BM25 (sparse retrieval) and FAISS (dense retrieval) using an Ensemble Retriever to enhance the relevance of retrieved documents.

In [11]:
query = "Punishment for murder under BNS"

docs = db.similarity_search(
    query,
    k=20
)

for d in docs:
    print(d.metadata)
    print(d.page_content[:500])
    print("="*50)

{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': '/content/legal_pdfs/Bharatiya Nyaya Sanhita.pdf', 'total_pages': 102, 'page': 33, 'page_label': '34', 'filename': 'Bharatiya Nyaya Sanhita.pdf', 'document_type': 'legal_act', 'law': 'Bharatiya Nyaya Sanhita', 'chunk_id': 219}
. 103.(1) Whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine. (2) When a group of five or more persons acting in concert commits murder on the ground of race, caste or community, sex, place of birth, language, personal belief or any other similar ground each member of such group shall be punished with death or with imprisonment for life, and shall also be liable to fine. 104.Whoever, being under sentence of imprisonment for life, commit
{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01

In [12]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 20

In [13]:
dense = db.as_retriever(
    search_kwargs={"k": 20}
)

In [14]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble = EnsembleRetriever(
    retrievers=[bm25, dense],
    weights=[0.4, 0.6]
)

In [15]:
docs = ensemble.invoke(
    "What is punishment under Section 103 BNS?"
)

In [16]:
for i, doc in enumerate(docs[:5]):
    print(f"\n===== Document {i+1} =====")
    print(doc.metadata)
    print(doc.page_content[:500])


===== Document 1 =====
{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': '/content/legal_pdfs/Bharatiya Nyaya Sanhita.pdf', 'total_pages': 102, 'page': 33, 'page_label': '34', 'filename': 'Bharatiya Nyaya Sanhita.pdf', 'document_type': 'legal_act', 'law': 'Bharatiya Nyaya Sanhita', 'chunk_id': 219}
. 103.(1) Whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine. (2) When a group of five or more persons acting in concert commits murder on the ground of race, caste or community, sex, place of birth, language, personal belief or any other similar ground each member of such group shall be punished with death or with imprisonment for life, and shall also be liable to fine. 104.Whoever, being under sentence of imprisonment for life, commit

===== Document 2 =====
{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozil

In [17]:
!pip install -q sentence-transformers

## 5. Reranking Setup

After initial retrieval, a Cross-Encoder reranker is used to further refine the relevance of the retrieved documents, ensuring the most pertinent information is passed to the LLM.

In [18]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [19]:
def remove_duplicates(docs):

    seen = set()

    unique_docs = []

    for doc in docs:

        key = (
            doc.metadata.get("filename"),
            doc.metadata.get("page")
        )

        if key not in seen:

            seen.add(key)

            unique_docs.append(doc)

    return unique_docs

In [20]:
def rerank_documents(query, docs, top_k=5):

    pairs = [[query, doc.page_content] for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    final_docs = []

    for doc, score in ranked[:top_k]:

        doc.metadata["rerank_score"] = float(score)

        final_docs.append(doc)

    return final_docs

In [21]:
query = "What is punishment under Section 103 BNS?"

retrieved_docs = ensemble.invoke(query)

top_docs = rerank_documents(
    query,
    retrieved_docs,
    top_k=5
)

In [22]:
for doc in top_docs:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("="*50)

{'producer': 'cairo 1.17.4 (https://cairographics.org)', 'creator': 'Mozilla Firefox 124.0.1', 'creationdate': '2024-04-01T12:07:18+05:30', 'source': '/content/legal_pdfs/Bharatiya Nyaya Sanhita.pdf', 'total_pages': 102, 'page': 33, 'page_label': '34', 'filename': 'Bharatiya Nyaya Sanhita.pdf', 'document_type': 'legal_act', 'law': 'Bharatiya Nyaya Sanhita', 'chunk_id': 219, 'rerank_score': 0.900134801864624}
. 103.(1) Whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine. (2) When a group of five or more persons acting in concert commits murder on the ground of race, caste or community, sex, place of birth, language, personal belief or any other similar ground each member of such group shall be punished with death or with imprisonment for life, and shall also be liable to fine. 104.Whoever, being under sentence of imprisonment for life, commit
{'producer': 'Acrobat Distiller 7.0 (Windows)', 'creator': 'PageMaker 7.0', 'creationda

## 6. LLM Integration and Response Generation

This final section integrates the retrieved and reranked documents with a Large Language Model (LLM) to generate informed answers based on the provided legal context. Helper functions are defined to manage context, prompts, and model interactions.

In [23]:
!pip install -q google-generativeai

In [24]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)


In [25]:
def retrieve_context(query, top_k=5):

    retrieved_docs = ensemble.invoke(query)

    retrieved_docs = remove_duplicates(
        retrieved_docs
    )

    top_docs = rerank_documents(
        query,
        retrieved_docs,
        top_k
    )

    return top_docs

In [44]:
def build_context(docs):

    context_parts = []

    for doc in docs:

        source = doc.metadata.get("filename", "Unknown")
        law = doc.metadata.get("law", "Unknown")
        page = doc.metadata.get("page", "Unknown")

        context_parts.append(
            f"""
Source: {source}
Law: {law}
Page: {page+1}

{doc.page_content}
"""
        )

    return "\n\n".join(context_parts)

In [26]:
def create_prompt(query, context):

    return f"""
You are Vidhi AI.

Follow these rules strictly:

1. Use ONLY provided context.
2. Do not use outside knowledge.
3. If answer is unavailable say:
"I could not find sufficient information in the retrieved legal documents."
4. Mention source law when relevant.
5. Mention page number if available.
6. Never fabricate sections.
7. Return concise and accurate legal information.

CONTEXT:

{context}

QUESTION:

{query}

ANSWER:
"""

In [38]:
def build_citations(docs):

    citations = []

    seen = set()

    for doc in docs:

        citation = (
            doc.metadata.get("filename"),
            doc.metadata.get("page")
        )

        if citation not in seen:

            seen.add(citation)

            citations.append({
                "source": doc.metadata.get("filename"),
                "page": doc.metadata.get("page")+1,
                "law": doc.metadata.get("law")
            })

    return citations

In [31]:
def build_response(answer, docs):

    confidence = round(
        sum(
            doc.metadata.get("rerank_score", 0)
            for doc in docs
        ) / len(docs),
        2
    )

    return {
        "answer": answer,
        "confidence": confidence,
        "citations": build_citations(docs)
    }

In [29]:
def generate_answer(query):

    docs = retrieve_context(query)

    if not docs:
        return {
            "answer": (
                "I could not find sufficient information "
                "in the retrieved legal documents."
            ),
            "confidence": 0.0,
            "citations": []
        }

    highest_score = max(
        doc.metadata.get("rerank_score", 0)
        for doc in docs
    )

    if highest_score < 0.4:
        return {
            "answer": (
                "The retrieved context does not contain "
                "enough relevant legal information."
            ),
            "confidence": round(highest_score, 2),
            "citations": []
        }

    context = build_context(docs)

    prompt = create_prompt(
        query=query,
        context=context
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text

    confidence = round(
        sum(
            doc.metadata.get("rerank_score", 0)
            for doc in docs
        ) / len(docs),
        2
    )

    return {
        "answer": answer,
        "confidence": confidence,
        "citations": build_citations(docs)
    }

In [45]:
answer = generate_answer(
    "What is punishment under Section 103 BNS?"
)


In [46]:
print(answer)

{'answer': 'Under Section 103 of the Bharatiya Nyaya Sanhita (Page: 34):\n\n1.  **Section 103(1)**: Whoever commits murder shall be punished with death or imprisonment for life, and shall also be liable to fine.\n2.  **Section 103(2)**: When a group of five or more persons acting in concert commits murder on the ground of race, caste or community, sex, place of birth, language, personal belief or any other similar ground, each member of such group shall be punished with death or with imprisonment for life, and shall also be liable to fine.', 'confidence': 0.28, 'citations': [{'source': 'Bharatiya Nyaya Sanhita.pdf', 'page': 34, 'law': 'Bharatiya Nyaya Sanhita'}, {'source': 'Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf', 'page': 182, 'law': 'Bharatiya Nagarik Suraksha Sanhita'}, {'source': 'Bharatiya_Nagarik_Suraksha_Sanhita,_2023.pdf', 'page': 181, 'law': 'Bharatiya Nagarik Suraksha Sanhita'}, {'source': 'Bharatiya Nyaya Sanhita.pdf', 'page': 42, 'law': 'Bharatiya Nyaya Sanhita'}, {'so

## 7. Example Query

This cell demonstrates how to use the `generate_answer` function with a sample legal query, showcasing the end-to-end functionality of the system.